# 02 — Pré-processamento
Monta o `ColumnTransformer` + `Pipeline` do scikit-learn garantindo que **nenhum leakage** ocorra entre treino e validação.

In [ ]:
import sys
sys.path.append('..')

import pandas as pd
import numpy as np

from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler, OneHotEncoder, FunctionTransformer
from sklearn.linear_model import Ridge

from src.feature_engineering import create_features, NAN_MEANS_NONE, ORDINAL_QUAL_COLS

## 1. Carregando e preparando os dados

In [ ]:
train_raw = pd.read_csv('../data/treino.csv')
test_raw  = pd.read_csv('../data/teste_publico.csv')

# Remove 2 outliers identificados no EDA
train_raw = train_raw[~((train_raw['GrLivArea'] > 4000) & (train_raw['SalePrice'] < 300_000))].copy()
print(f'Treino após remover outliers: {train_raw.shape}')

# Aplica feature engineering
train_fe = create_features(train_raw.drop(columns=['SalePrice']))
y = train_raw['SalePrice'].copy()

print(f'Shape após feature engineering: {train_fe.shape}')
print(f'Novas features: TotalSF, TotalBath, HouseAge, RemodAge, HasPool, HasGarage, HasFireplace, HasBsmt')

## 2. Classificando as colunas por tipo de tratamento

In [ ]:
# Após create_features, as colunas ordinais já estão numéricas (encodadas manualmente)
# e as NAN_MEANS_NONE já têm string 'None' — mas GarageType, Alley, Fence, MiscFeature ainda são categóricas

# Colunas já tratadas em create_features (não entram no transformer)
already_encoded = ORDINAL_QUAL_COLS + ['BsmtExposure', 'BsmtFinType1', 'BsmtFinType2', 'GarageFinish']
new_features = ['TotalSF', 'TotalBath', 'HouseAge', 'RemodAge', 'HasPool', 'HasGarage', 'HasFireplace', 'HasBsmt']

# Colunas categóricas: NaN agora é string 'None' — incluem GarageType, Alley, Fence, etc.
cat_none_cols = [
    c for c in NAN_MEANS_NONE
    if c in train_fe.columns and c not in already_encoded
    and train_fe[c].dtype == object
]

# Restante categóricas (NaN é dado realmente faltante)
cat_real_cols = [
    c for c in train_fe.select_dtypes(include='object').columns
    if c not in cat_none_cols and c not in already_encoded
]

# Colunas numéricas (inclui novas features criadas)
num_cols = [
    c for c in train_fe.select_dtypes(include=[np.number]).columns
    if c not in already_encoded
]

print(f'Numéricas        : {len(num_cols)} colunas')
print(f'Categ NaN=None   : {len(cat_none_cols)} colunas -> {cat_none_cols}')
print(f'Categ NaN=faltante: {len(cat_real_cols)} colunas -> {cat_real_cols}')
print(f'Já encodadas     : {len(already_encoded)} colunas')

## 3. Construindo o ColumnTransformer

In [ ]:
# Pipeline para numéricas: imputação por mediana + padronização
num_pipe = Pipeline([
    ('imputer', SimpleImputer(strategy='median')),
    ('scaler',  StandardScaler()),
])

# Pipeline para categóricas onde NaN = 'None' (já substituído por string em create_features)
# SimpleImputer aqui é precaução; OHE lida com 'None' como categoria válida
cat_none_pipe = Pipeline([
    ('imputer', SimpleImputer(strategy='constant', fill_value='None')),
    ('ohe',     OneHotEncoder(handle_unknown='ignore', sparse_output=False)),
])

# Pipeline para categóricas onde NaN é dado realmente faltante
cat_real_pipe = Pipeline([
    ('imputer', SimpleImputer(strategy='most_frequent')),
    ('ohe',     OneHotEncoder(handle_unknown='ignore', sparse_output=False)),
])

# Ordinal/already-encoded: só imputação por mediana (já são numéricas)
ord_pipe = Pipeline([
    ('imputer', SimpleImputer(strategy='median')),
])

preprocessor = ColumnTransformer([
    ('num',      num_pipe,      num_cols),
    ('cat_none', cat_none_pipe, cat_none_cols),
    ('cat_real', cat_real_pipe, cat_real_cols),
    ('ordinal',  ord_pipe,      already_encoded),
], remainder='drop')

print('ColumnTransformer construído.')

## 4. Testando o pipeline ponta-a-ponta

In [ ]:
from sklearn.model_selection import train_test_split

X_tr, X_val, y_tr, y_val = train_test_split(train_fe, y, test_size=0.2, random_state=42)

# Testa fit_transform
X_tr_proc = preprocessor.fit_transform(X_tr)
X_val_proc = preprocessor.transform(X_val)

print(f'Shape processado treino  : {X_tr_proc.shape}')
print(f'Shape processado validação: {X_val_proc.shape}')
print(f'NaNs no resultado: {np.isnan(X_tr_proc).sum()} (deve ser 0)')

In [ ]:
# Salva a lista de colunas para reutilizar nos outros notebooks
import json

col_config = {
    'num_cols': num_cols,
    'cat_none_cols': cat_none_cols,
    'cat_real_cols': cat_real_cols,
    'already_encoded': already_encoded,
}

with open('../src/col_config.json', 'w') as f:
    json.dump(col_config, f, indent=2)

print('Configuração de colunas salva em src/col_config.json')

## 5. Exportando o preprocessor
O `preprocessor` construído aqui será importado pelo notebook 03 e pelo `pipeline.py` final.

In [ ]:
# Função helper para construir o preprocessor (importável)
def build_preprocessor():
    """Retorna um preprocessor fit-ready, baseado na configuração de colunas salva."""
    import json
    with open('../src/col_config.json') as f:
        cfg = json.load(f)

    num_pipe = Pipeline([
        ('imputer', SimpleImputer(strategy='median')),
        ('scaler',  StandardScaler()),
    ])
    cat_none_pipe = Pipeline([
        ('imputer', SimpleImputer(strategy='constant', fill_value='None')),
        ('ohe',     OneHotEncoder(handle_unknown='ignore', sparse_output=False)),
    ])
    cat_real_pipe = Pipeline([
        ('imputer', SimpleImputer(strategy='most_frequent')),
        ('ohe',     OneHotEncoder(handle_unknown='ignore', sparse_output=False)),
    ])
    ord_pipe = Pipeline([
        ('imputer', SimpleImputer(strategy='median')),
    ])

    return ColumnTransformer([
        ('num',      num_pipe,               cfg['num_cols']),
        ('cat_none', cat_none_pipe,           cfg['cat_none_cols']),
        ('cat_real', cat_real_pipe,           cfg['cat_real_cols']),
        ('ordinal',  ord_pipe,                cfg['already_encoded']),
    ], remainder='drop')


print('Função build_preprocessor() definida e pronta para importar no notebook 03.')